# Cohort construction and demographic matching

Builds the study cohort and prepares the demographic variables used for matching and analysis.

This notebook accompanies [Stigmatizing Language in Gender-Expansive Patient Records: Corpus Development, Disparity Analysis, and Natural Language Processing-Based Detection Study](https://www.jmir.org/2026/1/e91089).

## Data and execution requirements

- Clinical note text and MIMIC identifiers are not included in this repository.
- Run this notebook only in an environment authorized to access MIMIC-IV and the credentialed annotation release.
- Set `GEP_DATA_DIR`, `GEP_MODEL_DIR`, `GEP_RESULTS_DIR`, and `GEP_FIGURES_DIR` as needed. By default, repository-local directories are used.
- The notebook outputs and execution counters have been removed from the public version.


In [ ]:
# Repository-local path configuration
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
FIGURES_DIR = Path(os.environ.get("GEP_FIGURES_DIR", PROJECT_ROOT / "figures")).resolve()

for directory in (MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
import os
import re
import string
import matplotlib.pyplot as plt

In [ ]:
df_gep = pd.read_csv(str(DATA_DIR / 'mimic_GEP_filtered.csv'))

In [ ]:
mimic = pd.read_csv(str(DATA_DIR / 'mimic-iv-note/2.2/note/discharge.csv'))

In [ ]:
patients = pd.read_csv(str(DATA_DIR / 'mimic-iv-demographic/patients.csv'))

In [ ]:
admissions = pd.read_csv(str(DATA_DIR / 'mimic-iv-demographic/admissions.csv'))

In [ ]:
df_merged = pd.merge(df_gep, admissions[['subject_id', 'insurance', 'language', 'marital_status', 'race']], on='subject_id', how='left')

In [ ]:
demographic = pd.merge(df_merged, patients[['gender', 'anchor_age', 'subject_id']], on='subject_id', how='left')

In [ ]:
demographic = demographic.drop_duplicates(subset=['note_id', 'text']).reset_index(drop=True)

In [ ]:
# Demographic analysis for the 'demographic' dataframe

# Display value counts for categorical columns
print("Value counts for demographic features:")
for col in ['gender', 'race', 'marital_status', 'language', 'insurance']:
    if col in demographic.columns:
        print(f"\n--- {col} ---")
        display(demographic[col].value_counts())

# Display descriptive statistics for numerical columns
print("\nDescriptive statistics for numerical features:")
display(demographic.describe())

## Prepare demographic matching variables


In [ ]:
def categorize_race(race):
    if race == 'WHITE':
        return 'WHITE'
    elif race == 'BLACK/AFRICAN AMERICAN':
        return 'BLACK/AFRICAN AMERICAN'
    elif 'ASIAN' in race:
        return 'ASIAN'
    elif 'HISPANIC/LATINO' in race or 'HISPANIC OR LATINO' in race:
        return 'HISPANIC/LATINO'
    else:
        return 'OTHER'

demographic['race_grouped'] = demographic['race'].apply(categorize_race)
demographic.head()

In [ ]:
demographic['language_grouped'] = demographic['language'].apply(lambda x: 'English' if x == 'English' else 'Non-English')
demographic.head()

In [ ]:
num_quantiles = 4
demographic['age_group'] = pd.qcut(demographic['anchor_age'], q=num_quantiles, labels=False, duplicates='drop')
print(f"\n--- age_group ({num_quantiles} quantiles) ---")
display(demographic['age_group'].value_counts())

In [ ]:
print("Distribution of newly formed demographic groups:")

print("\n--- race_grouped ---")
display(demographic['race_grouped'].value_counts())

print("\n--- language_grouped ---")
display(demographic['language_grouped'].value_counts())

print("\n--- age_group ---")
display(demographic['age_group'].value_counts())

print("\n--- Distribution by Race Group and Language Group ---")
display(pd.crosstab(demographic['race_grouped'], demographic['language_grouped']))

print("\n--- Distribution by Race Group and Age Group ---")
display(pd.crosstab(demographic['race_grouped'], demographic['age_group']))

print("\n--- Distribution by Language Group and Age Group ---")
display(pd.crosstab(demographic['language_grouped'], demographic['age_group']))

## Create eight stratified GEP subsets


In [ ]:
stratification_cols = ['race_grouped', 'language_grouped', 'age_group']
print("Stratification columns:")
print(stratification_cols)

In [ ]:
group_sizes = demographic.groupby(stratification_cols).size()
print("Size of each demographic group:")
display(group_sizes)

In [ ]:
total_individuals = len(demographic)
num_subsets = 8
target_subset_size = total_individuals // num_subsets
print(f"Total number of individuals: {total_individuals}")
print(f"Number of subsets: {num_subsets}")
print(f"Approximate target size for each subset: {target_subset_size}")

In [ ]:
subsets = [pd.DataFrame(columns=demographic.columns) for _ in range(num_subsets)]
sampled_indices = set()

for group_keys, group_size in group_sizes.items():
    # Prioritize race_grouped for proportional distribution
    race_group = group_keys[0]
    language_group = group_keys[1]
    age_group = group_keys[2]

    group_df = demographic[(demographic['race_grouped'] == race_group) &
                           (demographic['language_grouped'] == language_group) &
                           (demographic['age_group'] == age_group)]

    available_indices = group_df.index.difference(sampled_indices)
    available_size = len(available_indices)

    if available_size == 0:
        continue

    # Calculate proportional sample size for each subset, prioritizing race
    base_sample_per_subset = available_size // num_subsets
    remainder = available_size % num_subsets

    current_group_samples = []
    for i in range(num_subsets):
        sample_size = base_sample_per_subset + (1 if i < remainder else 0)
        if sample_size > 0:
            # Ensure we don't sample more than available
            sample_size = min(sample_size, len(available_indices))
            if sample_size > 0:
                sampled_group_indices = np.random.choice(available_indices, size=sample_size, replace=False)
                subsets[i] = pd.concat([subsets[i], demographic.loc[sampled_group_indices]])
                sampled_indices.update(sampled_group_indices)
                available_indices = available_indices.difference(sampled_group_indices)


# Display the size of each created subset
print("\nSize of each sampled subset:")
for i, subset in enumerate(subsets):
    print(f"Subset {i+1}: {len(subset)} individuals")

In [ ]:
for i, subset in enumerate(subsets):
    print(f"\n--- Subset {i+1} Distribution ---")

    print("\nValue counts for stratification columns:")
    for col in stratification_cols:
        print(f"\n--- {col} ---")
        display(subset[col].value_counts())

    print("\nCross-tabulations of stratification columns:")
    print("\n--- Race Group and Age Group ---")
    display(pd.crosstab(subset['race_grouped'], subset['age_group']))

    print("\n--- Race Group and Language Group ---")
    display(pd.crosstab(subset['race_grouped'], subset['language_grouped']))

    print("\n--- Language Group and Age Group ---")
    display(pd.crosstab(subset['language_grouped'], subset['age_group']))


## Prepare the comparison-note pool


In [ ]:
mimic_merged = pd.merge(mimic, admissions[['hadm_id', 'subject_id', 'insurance', 'language', 'marital_status', 'race']], on=['hadm_id', 'subject_id'], how='left')
mimic_merged = pd.merge(mimic_merged, patients[['subject_id', 'gender', 'anchor_age']], on='subject_id', how='left')
print("\nShape of the merged dataframe:")
print(mimic_merged.shape)

In [ ]:
mimic_merged = mimic_merged.drop_duplicates(subset=['note_id', 'text']).reset_index(drop=True)
print("Shape of the merged dataframe after removing duplicates:")
print(mimic_merged.shape)

In [ ]:
def categorize_race(race):
    if isinstance(race, str):
        if race == 'WHITE':
            return 'WHITE'
        elif race == 'BLACK/AFRICAN AMERICAN':
            return 'BLACK/AFRICAN AMERICAN'
        elif 'ASIAN' in race:
            return 'ASIAN'
        elif 'HISPANIC/LATINO' in race or 'HISPANIC OR LATINO' in race:
            return 'HISPANIC/LATINO'
        else:
            return 'OTHER'
    else:
        return 'UNKNOWN' # Handle missing values

mimic_merged['race_grouped'] = mimic_merged['race'].apply(categorize_race)
mimic_merged['language_grouped'] = mimic_merged['language'].apply(lambda x: 'English' if x == 'English' else 'Non-English')

# Assign age quartiles only to records with a nonmissing age.
mimic_merged_cleaned_age = mimic_merged.dropna(subset=['anchor_age'])

mimic_merged['age_group'] = pd.NA
mimic_merged.loc[mimic_merged_cleaned_age.index, 'age_group'] = pd.qcut(mimic_merged_cleaned_age['anchor_age'], q=4, labels=False, duplicates='drop')




In [ ]:
stratification_cols_mimic = ['race_grouped', 'language_grouped', 'age_group']
print("Stratification columns for mimic dataset:")
print(stratification_cols_mimic)

In [ ]:
target_mimic_subset_sizes = []
for subset in subsets:
    target_mimic_subset_sizes.append(len(subset))

print("Target subset sizes for mimic_merged dataset:")
print(target_mimic_subset_sizes)

In [ ]:
subsets_mimic = [pd.DataFrame(columns=mimic_merged.columns) for _ in range(num_subsets_mimic)]
sampled_indices_mimic = set()

race_proportions = mimic_merged['race_grouped'].value_counts(normalize=True)

for i in range(num_subsets_mimic):
    target_size_for_subset = target_mimic_subset_sizes[i]
    current_subset_indices = []

    # Iterate through each race group and determine the target number of samples for this subset
    for race_group, proportion in race_proportions.items():
        target_samples_for_race_group = int(target_size_for_subset * proportion)

        # Filter the available indices for the current race group
        race_group_indices = mimic_merged[mimic_merged['race_grouped'] == race_group].index
        available_indices_for_race_group = list(race_group_indices.difference(sampled_indices_mimic))

        sample_size = min(target_samples_for_race_group, len(available_indices_for_race_group))

        if sample_size > 0:
            sampled_indices_from_group = np.random.choice(available_indices_for_race_group, size=sample_size, replace=False)
            current_subset_indices.extend(sampled_indices_from_group)
            sampled_indices_mimic.update(sampled_indices_from_group)

    np.random.shuffle(current_subset_indices)
    subsets_mimic[i] = pd.concat([subsets_mimic[i], mimic_merged.loc[current_subset_indices]])


print("\nSize of each sampled subset from the merged mimic dataset:")
for i, subset in enumerate(subsets_mimic):
    print(f"Subset {i+1}: {len(subset)} individuals")

## Check matched-subset distributions


In [ ]:
for i, subset in enumerate(subsets_mimic):
    print(f"\n--- Subset {i+1} Distribution ---")

    print("\nValue counts for stratification columns:")
    for col in stratification_cols_mimic:
        print(f"\n--- {col} ---")
        display(subset[col].value_counts())

    print("\nCross-tabulations of stratification columns:")
    print("\n--- Race Group and Age Group ---")
    display(pd.crosstab(subset['race_grouped'], subset['age_group']))

    print("\n--- Race Group and Language Group ---")
    display(pd.crosstab(subset['race_grouped'], subset['language_grouped']))

    print("\n--- Language Group and Age Group ---")
    display(pd.crosstab(subset['language_grouped'], subset['age_group']))

## Combine and save the matched subsets


In [ ]:
print("Comparison of Demographic Distribution between Demographic Subsets and Mimic Subsets")

for i in range(len(subsets)):
    print(f"\n--- Subset {i+1} Distribution Comparison ---")

    print("\n--- Demographic Subset ---")
    print("\nValue counts for stratification columns:")
    for col in stratification_cols:
        print(f"\n--- {col} ---")
        display(subsets[i][col].value_counts())

    print("\nCross-tabulations of stratification columns:")
    print("\n--- Race Group and Age Group ---")
    display(pd.crosstab(subsets[i]['race_grouped'], subsets[i]['age_group']))

    print("\n--- Race Group and Language Group ---")
    display(pd.crosstab(subsets[i]['race_grouped'], subsets[i]['language_grouped']))

    print("\n--- Language Group and Age Group ---")
    display(pd.crosstab(subsets[i]['language_grouped'], subsets[i]['age_group']))

    print("\n--- Mimic Subset ---")
    print("\nValue counts for stratification columns:")
    for col in stratification_cols_mimic:
        print(f"\n--- {col} ---")
        display(subsets_mimic[i][col].value_counts())

    print("\nCross-tabulations of stratification columns:")
    print("\n--- Race Group and Age Group ---")
    display(pd.crosstab(subsets_mimic[i]['race_grouped'], subsets_mimic[i]['age_group']))

    print("\n--- Race Group and Language Group ---")
    display(pd.crosstab(subsets_mimic[i]['race_grouped'], subsets_mimic[i]['language_grouped']))

    print("\n--- Language Group and Age Group ---")
    display(pd.crosstab(subsets_mimic[i]['language_grouped'], subsets_mimic[i]['age_group']))

In [ ]:
combined_subsets = []

for i in range(len(subsets)):
    demographic_subset = subsets[i]
    mimic_subset = subsets_mimic[i]

    combined_subset = pd.concat([demographic_subset, mimic_subset], ignore_index=True, sort=False)

    combined_subsets.append(combined_subset)


print(f"Created {len(combined_subsets)} combined subsets.")
print("\nShape of the first combined subset:")
display(combined_subsets[0].shape)


In [ ]:
cleaned_combined_subsets = []

for subset in combined_subsets:
    subset_cleaned = subset.dropna(axis=1, how='all')
    cleaned_combined_subsets.append(subset_cleaned)

combined_subsets = cleaned_combined_subsets

print(f"Cleaned {len(combined_subsets)} combined subsets by removing all-NaN columns.")
print("\nShape of the first cleaned combined subset:")
display(combined_subsets[0].shape)


In [ ]:
columns_to_drop = ['hadm_id', 'note_type', 'note_seq', 'charttime', 'storetime']
cleaned_combined_subsets_further = []

for subset in combined_subsets:
    subset_cleaned = subset.drop(columns=columns_to_drop, errors='ignore')
    cleaned_combined_subsets_further.append(subset_cleaned)

combined_subsets = cleaned_combined_subsets_further

print(f"Removed specified columns from {len(combined_subsets)} combined subsets.")
print("\nShape of the first combined subset after removing columns:")
display(combined_subsets[0].shape)


In [ ]:
print("Shape of each subset from the demographic dataset:")
for i, subset in enumerate(subsets):
    print(f"Demographic Subset {i+1}: {subset.shape}")

print("\nShape of each subset from the mimic dataset:")
for i, subset in enumerate(subsets_mimic):
    print(f"Mimic Subset {i+1}: {subset.shape}")

In [ ]:
columns_to_remove = ['transgender', 'transwoman', 'transman', 'transmasculine', 'transfeminine', 'queer', 'intersex', 'genderqueer', 'feminized', 'masculine', 'ftm', 'mtf', 'transexual', 'agab', 'feminization', 'row_sum']
cleaned_combined_subsets_final = []

for subset in combined_subsets:
    subset_cleaned = subset.drop(columns=columns_to_remove, errors='ignore')
    cleaned_combined_subsets_final.append(subset_cleaned)

combined_subsets = cleaned_combined_subsets_final

print(f"Removed specified columns from {len(combined_subsets)} combined subsets.")
print("\nShape of the first combined subset after removing columns:")
display(combined_subsets[0].shape)


In [ ]:
import os

output_dir = str(DATA_DIR) + os.sep

os.makedirs(output_dir, exist_ok=True)

for i, subset in enumerate(combined_subsets):
    filename = f'GEP_Demographic_Based_Sample_Subset_{i+1}.csv'
    output_path = os.path.join(output_dir, filename)

    subset.to_csv(output_path, index=False)
    print(f"Subset {i+1} saved to: {output_path}")

print("\nAll combined subsets have been saved with the specified naming convention.")